# 04 - SARIMA training for ds003029

Notebook nay dung cho nhanh `bim`.

Muc tieu:
1. Build multirun window features.
2. Train SARIMA thuan theo tung run.
3. Danh gia bang chronological train/test split.
4. Ve plot tong hop theo format 3 panel.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "ds003029_eda").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing src/ds003029_eda")


repo_root = find_repo_root()
sys.path.insert(0, str(repo_root / "src"))
repo_root

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from ds003029_eda.paths import get_paths
from ds003029_eda.sarima_training import run_sarima_training
from ds003029_eda.window_features_multirun import WindowingConfig, build_multirun_window_features

sns.set_style("whitegrid")

In [ ]:
paths = get_paths(repo_root)
paths

In [ ]:
features_df, info_df = build_multirun_window_features(
    paths=paths,
    config=WindowingConfig(window_sec=2.0, step_sec=1.0, max_channels=16),
    output_name="ds003029_window_features_multirun_full.csv",
    output_info_name="ds003029_windowing_multirun_full_info.csv",
)

display(features_df.head())
display(info_df.head())

In [ ]:
prepared, metrics_df = run_sarima_training(
    feature_path="ds003029_window_features_multirun_full.csv",
    output_subdir="sarima_training",
    paths=paths,
    min_total_obs=36,
    min_test_obs=12,
    test_fraction=0.2,
)

display(metrics_df)

In [ ]:
ok_metrics = metrics_df[metrics_df["status"] == "ok"].copy()
display(
    ok_metrics[
        [
            "series_id",
            "n_train",
            "n_test",
            "selected_order",
            "selected_seasonal_order",
            "aic_train",
            "rmse_test",
            "mae_test",
            "mape_test_pct",
            "r2_test",
            "ljungbox_pvalue_train_residual",
        ]
    ]
)

In [ ]:
predictions_path = paths.outputs_dir / "sarima_training" / "sarima_all_predictions.csv"
predictions_df = pd.read_csv(predictions_path)
display(predictions_df.head())

In [ ]:
selected_series_id = ok_metrics.iloc[0]["series_id"]
series_df = predictions_df[predictions_df["series_id"] == selected_series_id].copy()
series_df = series_df.sort_values("t_mid_s").reset_index(drop=True)
train_mask = series_df["split"] == "train"
test_mask = series_df["split"] == "test"
series_df["abs_error"] = (series_df["rms"] - series_df["sarima_pred"]).abs()
selected_series_id

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 16))
fig.suptitle(
    f"SARIMA detailed fit report - {Path(str(selected_series_id)).stem}",
    fontsize=16,
    fontweight="bold",
    y=0.97,
)

axes[0].plot(series_df["t_mid_s"], series_df["rms"], label="Actual RMS", color="#1f77b4", linewidth=1.8)
axes[0].plot(series_df["t_mid_s"], series_df["sarima_pred"], label="SARIMA fitted", color="#d62728", linestyle="--", linewidth=1.6)
split_time = series_df.loc[test_mask, "t_mid_s"].min()
axes[0].axvline(split_time, color="gray", linestyle=":", linewidth=1.2, label="train/test split")
axes[0].set_title("1. Full-sequence fit: actual vs fitted values", fontsize=13)
axes[0].set_ylabel("RMS")
axes[0].legend(loc="upper left")
axes[0].grid(True, linestyle=":", alpha=0.7)

test_plot = series_df.loc[test_mask, ["t_mid_s", "abs_error"]].copy()
axes[1].plot(test_plot["t_mid_s"], test_plot["abs_error"], color="purple", linewidth=1.5)
axes[1].fill_between(test_plot["t_mid_s"], test_plot["abs_error"], color="purple", alpha=0.18)
axes[1].set_title("2. Absolute fitting error over time", fontsize=13)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("|Actual - Fitted|")
axes[1].grid(True, linestyle=":", alpha=0.7)

scatter_df = series_df.loc[test_mask, ["rms", "sarima_pred"]].copy()
sns.scatterplot(
    data=scatter_df,
    x="rms",
    y="sarima_pred",
    ax=axes[2],
    color="#ff7f0e",
    edgecolor="k",
    alpha=0.75,
    s=45,
)
min_val = min(scatter_df["rms"].min(), scatter_df["sarima_pred"].min())
max_val = max(scatter_df["rms"].max(), scatter_df["sarima_pred"].max())
axes[2].plot([min_val, max_val], [min_val, max_val], color="red", linestyle="--", linewidth=1.5, label="Ideal fit (y=x)")
axes[2].set_title("3. Actual vs SARIMA fitted scatter", fontsize=13)
axes[2].set_xlabel("Actual RMS")
axes[2].set_ylabel("SARIMA fitted RMS")
axes[2].legend(loc="upper left")
axes[2].grid(True, linestyle=":", alpha=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()